In [ ]:
# reading the output of zeroed-out interaction inference

import uproot
import os
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
import subprocess

In [ ]:
arrays = {}

In [ ]:
# move the .roots to cwd
print(f'Current directory: {os.getcwd()}')
for file in os.listdir('./'):
    if file.endswith('.root'):
        print(f"Reading file: {file}") 
        with uproot.open(file) as file:
            tree = file["Events"]
            for key in tree.arrays(library='np').keys():
                if key not in arrays:
                    arrays[key] = []
                arrays[key]=np.concatenate([arrays[key],tree.arrays(library='np')[key]])

print(arrays.keys())

In [ ]:
# scores are for standard inference on fully trained JC model with interaction
# load from .npys moved to this directory
subprocess.run(['sudo cp /moe-interpretability-pv/ParT_full_output_balanced/batched_preds/*.npy ./'], shell=True)
scores = np.array([])
for file in sorted(os.listdir('./')):
    if file.startswith('predictions_batch_') and file.endswith('.npy'):
        print(f"Reading predictions from: {file}")
        predictions = np.load(file)
        print(f"Predictions shape: {predictions.shape}")
        scores = np.concatenate([scores, predictions])

In [ ]:
# zero_u_scores are for inference on the same model but with all interaction features zeroed out
zero_u_scores = np.stack([arrays[i] for i in arrays.keys() if i.startswith('score_')], axis=1)
#print(zero_u_scores.shape)
print(zero_u_scores[0:4,:])

# check that they sum to 1
#print(np.sum(zero_u_scores, axis=1))

y_pred = np.argmax(zero_u_scores, axis=1)
print(y_pred[:10])
y_true = arrays['_label_']
print(y_true[:10])
print(np.where(y_true != 8*np.ones(len(y_true))))

#print(y_pred[:10])
acc = np.sum(y_pred == y_true) / len(y_true)
print(f"Accuracy from zero_u_scores: {acc}")

In [ ]:
print(y_true[:10])

In [ ]:
# Getting the ROC curve for Hbb vs non-Hbb
from sklearn.metrics import confusion_matrix, roc_curve, auc
cnf_matrix = confusion_matrix(y_true, y_pred)
fpr, tpr, thresholds = roc_curve((y_true == 1), zero_u_scores[:,1])
#print("Confusion Matrix:")
#print(cnf_matrix)
norm_cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
#print(f"normalized confusion matrix:\n{norm_cnf_matrix}")
#grid plot of cnf matrix
plt.figure(figsize=(8,6))
plt.imshow(norm_cnf_matrix, vmin=0, vmax=1, interpolation='nearest', cmap=plt.cm.Blues)
#show value of each cell
for i in range(cnf_matrix.shape[0]):
    for j in range(cnf_matrix.shape[1]):
        plt.text(j, i, f"{norm_cnf_matrix[i, j]:.2f}",
                 horizontalalignment="center", color="white" if norm_cnf_matrix[i, j] > 0.5 else "black")
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(cnf_matrix))
labels = ['QCD', 'Hbb', 'Hcc', 'Hgg', 'H4q', 'Hqql', 'Zqq', 'Wqq', 'Tbqq', 'Tbl']
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

def rejection(fpr, target_tpr, zero_u_scores, labels):
    # Find the threshold that gives the target TPR
    idx = np.argmax(tpr >= target_tpr)
    threshold = thresholds[idx]
    # Calculate the rejection (1 / FPR) at this threshold
    fpr_at_threshold = fpr[idx]
    if fpr_at_threshold == 0:
        return float('inf')  # Infinite rejection if FPR is 0
    return 1.0 / fpr_at_threshold

In [ ]:
print(fpr, tpr, thresholds)

In [ ]:
# all-classes AUC
from sklearn.metrics import roc_auc_score

# Calculate multi-class AUC using one-vs-rest approach
# This gives us the macro-average AUC across all classes
all_classes_auc = roc_auc_score(y_true, zero_u_scores, multi_class='ovr', average='macro')
print(f"All-classes macro-averaged AUC: {all_classes_auc:.4f}")

# Also calculate weighted average (accounts for class imbalance, though classes are balanced here)
weighted_auc = roc_auc_score(y_true, zero_u_scores, multi_class='ovr', average='weighted')
print(f"All-classes weighted-averaged AUC: {weighted_auc:.4f}")

# For comparison, show individual class AUCs again
print("\nIndividual class AUCs:")
class_names = ['QCD', 'H→bb', 'H→cc', 'H→gg', 'H→4q', 'H→qqℓ', 'Z→qq', 'W→qq', 'T→bqq', 'T→bℓ']
for i, class_name in enumerate(class_names):
    y_binary = (y_true == i).astype(int)
    class_auc = roc_auc_score(y_binary, zero_u_scores[:, i])
    print(f"{class_name}: {class_auc:.4f}")

In [ ]:
# print roc curve
plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % auc(fpr, tpr))
plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

In [ ]:
# overall accuracy, TPR, FPR
num_events = len(arrays['_label_'])
num_correct = np.sum(arrays['_label_'] == arrays['predicted_label'])
accuracy = num_correct / num_events


In [ ]:
print(arrays['_label_'])
#print(arrays['predicted_label'])
print(arrays['label_Hbb'])
print(arrays['score_label_Hbb'])

In [ ]:
# Getting the ROC curve for Hbb vs non-Hbb
from sklearn.metrics import confusion_matrix, roc_curve, auc
cnf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cnf_matrix)

In [ ]:
def rejection(fpr, target_tpr, zero_u_scores, labels):
    # Find the threshold that gives the target TPR
    idx = np.argmax(tpr >= target_tpr)
    threshold = thresholds[idx]
    # Calculate the rejection (1 / FPR) at this threshold
    fpr_at_threshold = fpr[idx]
    if fpr_at_threshold == 0:
        return float('inf')  # Infinite rejection if FPR is 0
    return 1.0 / fpr_at_threshold

In [ ]:
from sklearn.metrics import roc_curve

# rejection for each label (target TPR = 0.5 except for Hlvqq TPR = 0.99 and Tblv TPR = 0.995)
target_tprs = [0.5]*10
target_tprs[5] = 0.99  # Hlvqq
target_tprs[9] = 0.995 # Tblv
rejections = []

label_to_name = {
    0: 'QCD',
    1: 'Hbb',
    2: 'Hcc',
    3: 'Hgg',
    4: 'H4q',
    5: 'Hqql',
    6: 'Zqq',
    7: 'Wqq',
    8: 'Tbqq',
    9: 'Tbl'
}

for i in range(10):
    fpr, tpr, thresholds = roc_curve((y_true == i), zero_u_scores[:,i])
    rej = rejection(fpr, target_tprs[i], zero_u_scores[:,i], y_true)
    rejections.append(rej)
    print(f"Rejection for class {label_to_name[i]} at TPR={target_tprs[i]}: {rej:.2f}")